# CRIVO — Salto do 17M, fase 3: o realizador do quadro

**Ideia** (`docs/pesquisa_mente_artificial_20261012.md`): no cérebro, linguagem e pensamento são sistemas diferentes. No CRIVO, o raciocínio fica nos módulos e no quadro (`estado_interno.py`); o Transformer 17M vira a **rede da linguagem**: recebe o que o quadro decidiu e diz isso em português, no jeito de ser de `IDENTIDADE.md`.

**Entrada:** `<documento> quadro <fim> <usuario> mensagem <fim> <assistente>` — o quadro traz o ato (explicar, não sei, acolher, registrar, recordar, aceitar correção, recusar, esclarecer, falar de si, saudar) e o conteúdo (fatos verificados, nome, sentimento, motivo). **Saída:** a fala, terminando em `<fim>`.

**Dados** (congelados no repositório, conferidos por SHA): fatos do acervo do CRIVO, nomes inventados e respostas no tom da identidade. O blind usa assuntos e nomes nunca vistos e uma formulação de mensagem reservada por ato. Toda resposta-alvo passa nas mesmas métricas da avaliação.

**Treino:** parte da base exportada pela fase 2 (`CRIVO/transformer-17m-continuo/base/`, só leitura); ajuste fino do modelo inteiro, com replay do texto do pré-treino para não esquecer o português geral; o melhor ponto é escolhido pelo dev, entre os que preservam a perda de linguagem (≤ 105% da base).

**Gates no blind:** tudo ok ≥ 85%, fatos fiéis ≥ 90%, nome exato ≥ 90%, termina ≥ 95%, sem nome inventado ≥ 95%, cada ato ≥ 70%, perda de linguagem ≤ 105% da base. Mesmo passando: porta conversacional pendente, `ativo_no_chat=false`, `promovido=false`. Saída em `CRIVO/transformer-17m-fase3/`.

> **Ambiente de execução → Executar tudo** em um Colab com GPU, depois que a fase 2 exportar a base.


In [ ]:
# Célula 0 — GPU, Drive e base da fase 2
import os, sys, json, shutil, subprocess, hashlib, time, math, random
from pathlib import Path
from datetime import datetime
import numpy as np
import torch
import torch.nn.functional as F

if not torch.cuda.is_available():
    raise SystemExit("ERRO: GPU não disponível. Ambiente de execução → Alterar tipo de ambiente → GPU.")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch", torch.__version__, flush=True)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# A pasta do CRIVO pode ter outro nome nesta conta (atalho ao lado de uma pasta antiga):
# usa a primeira pasta de "Meu Drive" que tem a base exportada pela fase 2.
MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "transformer-17m-continuo" / "base" / "pesos.pt").is_file()), MEU / "CRIVO")
print("Pasta do CRIVO:", CRIVO, flush=True)
BASE = CRIVO / "transformer-17m-continuo" / "base"   # exportada pela fase 2 (só leitura)
CACHE = CRIVO / "dados-17m"                            # texto do pré-treino: replay e preservação
SAIDA = CRIVO / "transformer-17m-fase3"                # tudo desta fase fica aqui
for p in (BASE / "pesos.pt", BASE / "tokenizer.json", CACHE / "treino.bin", CACHE / "validacao.bin"):
    if not p.is_file():
        raise SystemExit(f"ERRO: não encontrei {p}. A fase 3 parte da base exportada pela fase 2.")
dispositivo = torch.device("cuda")
SEED = 20261012
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Base:", BASE, flush=True)


In [ ]:
# Célula 1 — código e dados congelados da fase 3
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"], check=True)
EXP = REPO / "experimentos" / "fase3_realizador_20261012"

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

ESPERADO = {"dados_fase3.json": "e1a83d4b6af1309f29280108005214f9bc1e5a2fab7651b63cbfd38b8967bced", "formato_fase3.py": "d1b4ab176c89b29f4042d83ddd7df32f5c24ca09d253603992d836ce17d96abb"}
for nome, esperado in ESPERADO.items():
    if sha256(EXP / nome) != esperado:
        raise SystemExit(f"ERRO: {nome} difere do congelado")
for p in (str(REPO), str(EXP)):
    if p not in sys.path:
        sys.path.insert(0, p)
import linguagem_profunda as lp
import formato_fase3 as ff
DADOS = json.loads((EXP / "dados_fase3.json").read_text(encoding="utf-8"))
print("commit:", COMMIT, "| casos:", {k: len(v) for k, v in DADOS.items()}, flush=True)


In [ ]:
# Célula 2 — carregar a base, tokenizar os casos e preparar o replay
LOCAL = Path("/content/fase3_base")
if LOCAL.exists():
    shutil.rmtree(LOCAL)
shutil.copytree(BASE, LOCAL)
SHA_BASE = sha256(LOCAL / "pesos.pt")
modelo, tokenizer, estado_base = lp.carregar(LOCAL, dispositivo="cpu")
cfg = modelo.config
modelo = modelo.to(dispositivo)
ID = {s: tokenizer.token_to_id(s) for s in lp.ESPECIAIS}
PROIBIDOS = [ID[s] for s in lp.ESPECIAIS if s != "<fim>"]

def preparar(casos):
    out = []
    for c in casos:
        prompt, resp = ff.montar_ids(lp, tokenizer, c["quadro"], c["mensagem"], c["resposta"], cfg.contexto)
        out.append({**c, "prompt_ids": prompt, "resp_ids": resp})
    return out

TREINO, DEV, BLIND = (preparar(DADOS[k]) for k in ("treino", "dev", "blind"))
DADOS_LM = Path("/content/dados-17m")
DADOS_LM.mkdir(exist_ok=True)
for n in ("treino.bin", "validacao.bin"):
    if not (DADOS_LM / n).is_file():
        shutil.copyfile(CACHE / n, DADOS_LM / n)
LM_TREINO = np.memmap(DADOS_LM / "treino.bin", dtype=np.uint16, mode="r")
LM_VALID = np.memmap(DADOS_LM / "validacao.bin", dtype=np.uint16, mode="r")
print(f"base sha256 {SHA_BASE} | passo {estado_base.get('passo')} | {sum(p.numel() for p in modelo.parameters()):,} parâmetros", flush=True)
print("treino", len(TREINO), "| dev", len(DEV), "| blind", len(BLIND), "| texto do pré-treino:", len(LM_TREINO), "tokens", flush=True)


In [ ]:
# Célula 3 — geração, avaliação e perda de linguagem (preservação do português geral)
@torch.no_grad()
def gerar(prompt_ids, max_novos=120):
    modelo.eval()
    ids, saida = list(prompt_ids), []
    for _ in range(min(max_novos, cfg.contexto - len(ids))):
        x = torch.tensor([ids], device=dispositivo)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            lg = modelo(x)[0][0, -1].float()
        lg[PROIBIDOS] = -1e9
        nxt = int(lg.argmax())
        if nxt == ID["<fim>"]:
            return saida, True
        saida.append(nxt)
        ids.append(nxt)
    return saida, False

def avaliar(casos, rotulo):
    linhas, t0 = [], time.time()
    for i, c in enumerate(casos, 1):
        g, fim = gerar(c["prompt_ids"])
        texto = tokenizer.decode(g)
        linhas.append({"ato": c["ato"], "mensagem": c["mensagem"], "alvo": c["resposta"], "saida": texto.strip(),
                       **ff.avaliar_saida(c, texto, fim)})
        if i == 1 or i % 100 == 0 or i == len(casos):
            print(f"  {rotulo}: {i}/{len(casos)} ({time.time()-t0:.0f}s)", flush=True)
    return linhas

@torch.no_grad()
def perda_lm(lotes=40, tamanho=16):
    modelo.eval()
    g = np.random.default_rng(SEED)
    perdas = []
    for _ in range(lotes):
        ini = g.integers(0, len(LM_VALID) - cfg.contexto - 1, size=tamanho)
        x = torch.from_numpy(np.stack([LM_VALID[i:i + cfg.contexto] for i in ini]).astype(np.int64)).to(dispositivo)
        y = torch.from_numpy(np.stack([LM_VALID[i + 1:i + 1 + cfg.contexto] for i in ini]).astype(np.int64)).to(dispositivo)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            perdas.append(modelo(x, y)[1].item())
    return float(np.mean(perdas))

def mostrar(res, rotulo):
    f = lambda v: "  —  " if v is None else f"{100*v:5.1f}%"
    print(f"\n--- {rotulo}: tudo ok {f(res['tudo_ok'])} | termina {f(res['terminou'])} | fatos fiéis {f(res['fatos_fieis'])} "
          f"| nome exato {f(res['nome_exato'])} | ato ok {f(res['ato_ok'])} | sem nome inventado {f(res['sem_nome_inventado'])}", flush=True)
    for ato, r in res["por_ato"].items():
        print(f"    {ato:17s} n={r['n']:3d} tudo ok {f(r['tudo_ok'])}", flush=True)

print("\n=== BASELINE: base da fase 2, sem ajuste ===", flush=True)
PERDA_LM_BASE = perda_lm()
BASE_BLIND_LINHAS = avaliar(BLIND, "base/blind")
BASE_BLIND = ff.resumir(BASE_BLIND_LINHAS)
mostrar(BASE_BLIND, "base no blind")
print(f"perda de linguagem (validação do pré-treino): {PERDA_LM_BASE:.4f}", flush=True)


In [ ]:
# Célula 4 — ajuste fino do modelo inteiro: realizar o quadro + replay do pré-treino
PASSOS = 1500
LOTE_SFT = 24
LOTE_REPLAY = 8
PESO_REPLAY = 0.5
LR = 1e-4
AQUECIMENTO = 100
AVALIAR_A_CADA = 250
PACIENCIA = 3
LIMITE_PRESERVACAO = 1.05   # perda de linguagem final ≤ 105% da base

def lote_sft(casos, rr):
    esc = rr.sample(casos, LOTE_SFT)
    seqs = [c["prompt_ids"] + c["resp_ids"] for c in esc]
    T = max(map(len, seqs))
    x = torch.full((len(seqs), T), ID["<pad>"], dtype=torch.long)
    y = torch.full((len(seqs), T), -100, dtype=torch.long)
    for i, (c, s) in enumerate(zip(esc, seqs)):
        x[i, :len(s)] = torch.tensor(s)
        P = len(c["prompt_ids"])
        y[i, P - 1:len(s) - 1] = torch.tensor(c["resp_ids"])
    return x.to(dispositivo), y.to(dispositivo)

def lote_replay(rng):
    ini = rng.integers(0, len(LM_TREINO) - cfg.contexto - 1, size=LOTE_REPLAY)
    x = np.stack([LM_TREINO[i:i + cfg.contexto] for i in ini]).astype(np.int64)
    y = np.stack([LM_TREINO[i + 1:i + 1 + cfg.contexto] for i in ini]).astype(np.int64)
    return torch.from_numpy(x).to(dispositivo), torch.from_numpy(y).to(dispositivo)

decai = [p for n, p in modelo.named_parameters() if p.dim() >= 2]
sem_decai = [p for n, p in modelo.named_parameters() if p.dim() < 2]
otim = torch.optim.AdamW([{"params": decai, "weight_decay": 0.1}, {"params": sem_decai, "weight_decay": 0.0}],
                         lr=LR, betas=(0.9, 0.95))
escala = torch.amp.GradScaler("cuda")

def taxa(p):
    if p < AQUECIMENTO:
        return LR * (p + 1) / AQUECIMENTO
    prog = (p - AQUECIMENTO) / max(1, PASSOS - AQUECIMENTO)
    return LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * prog)))

rr, rng = random.Random(SEED + 1), np.random.default_rng(SEED + 2)
historico, melhor, sem_melhora, t0 = [], None, 0, time.time()
DEV_MONITOR = DEV  # 325 casos, todos os atos
for passo in range(1, PASSOS + 1):
    modelo.train()
    for g in otim.param_groups:
        g["lr"] = taxa(passo)
    xs, ys = lote_sft(TREINO, rr)
    xr, yr = lote_replay(rng)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        perda_sft = modelo(xs, ys)[1]
        perda_rep = modelo(xr, yr)[1]
        perda = perda_sft + PESO_REPLAY * perda_rep
    otim.zero_grad(set_to_none=True)
    escala.scale(perda).backward()
    escala.unscale_(otim)
    torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
    escala.step(otim)
    escala.update()
    if passo % 50 == 0 or passo == 1:
        print(f"passo {passo:4d}/{PASSOS} sft {perda_sft.item():.3f} replay {perda_rep.item():.3f} lr {taxa(passo):.2e} "
              f"({time.time()-t0:.0f}s)", flush=True)
    if passo % AVALIAR_A_CADA == 0 or passo == PASSOS:
        res = ff.resumir(avaliar(DEV_MONITOR, f"dev@{passo}"))
        plm = perda_lm()
        preserva = plm <= PERDA_LM_BASE * LIMITE_PRESERVACAO
        historico.append({"passo": passo, "dev": res, "perda_lm": plm, "preserva": preserva})
        print(f"  DEV passo {passo}: tudo ok {100*res['tudo_ok']:.1f}% | perda LM {plm:.4f} (base {PERDA_LM_BASE:.4f}) "
              f"| preserva={preserva}", flush=True)
        if preserva and (melhor is None or res["tudo_ok"] > melhor["tudo_ok"] + 1e-4):
            melhor = {"passo": passo, "tudo_ok": res["tudo_ok"],
                      "estado": {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}}
            sem_melhora = 0
        else:
            sem_melhora += 1
            if sem_melhora >= PACIENCIA:
                print("  EARLY STOP", flush=True)
                break
if melhor is None:
    raise SystemExit("ERRO: nenhum ponto do treino preservou o português geral; nada será salvo como candidato.")
modelo.load_state_dict(melhor["estado"])
print(f"melhor passo pelo dev: {melhor['passo']} (tudo ok {100*melhor['tudo_ok']:.1f}%)", flush=True)


In [ ]:
# Célula 5 — avaliação final no blind (assuntos e nomes nunca vistos), gates e salvamento
FINAL_LINHAS = avaliar(BLIND, "final/blind")
FINAL = ff.resumir(FINAL_LINHAS)
PERDA_LM_FINAL = perda_lm()
mostrar(BASE_BLIND, "base no blind")
mostrar(FINAL, "realizador no blind")
print(f"perda de linguagem: base {PERDA_LM_BASE:.4f} → realizador {PERDA_LM_FINAL:.4f}", flush=True)

GATES = {
    "tudo_ok>=0.85": FINAL["tudo_ok"] >= 0.85,
    "fatos_fieis>=0.90": FINAL["fatos_fieis"] >= 0.90,
    "nome_exato>=0.90": FINAL["nome_exato"] >= 0.90,
    "terminou>=0.95": FINAL["terminou"] >= 0.95,
    "sem_nome_inventado>=0.95": FINAL["sem_nome_inventado"] >= 0.95,
    "cada_ato_tudo_ok>=0.70": all(r["tudo_ok"] >= 0.70 for r in FINAL["por_ato"].values()),
    "perda_lm<=105%_base": PERDA_LM_FINAL <= PERDA_LM_BASE * LIMITE_PRESERVACAO,
}
GATE_PASS = all(GATES.values())
print(json.dumps(GATES, ensure_ascii=False, indent=1), flush=True)
print("\nExemplos (blind):", flush=True)
for l in FINAL_LINHAS[:12]:
    print(f"[{l['ato']}] {l['mensagem']}\n   → {l['saida']}\n   ok={l['tudo_ok']}", flush=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DEST = SAIDA / f"fase3_{stamp}"
DEST.mkdir(parents=True, exist_ok=False)
estado_cpu = {k: v.detach().cpu() for k, v in modelo.state_dict().items()}
torch.save({"versao": lp.VERSAO, "modelo": estado_cpu, "config": estado_base["config"],
            "passo": melhor["passo"],
            "execucao": {"tokenizer_sha256": sha256(LOCAL / "tokenizer.json"), "pesos_externos": False,
                         "base_sha256": SHA_BASE, "protocolo": "fase3_realizador", "commit": COMMIT}},
           DEST / "pesos.pt")
shutil.copyfile(LOCAL / "tokenizer.json", DEST / "tokenizer.json")
resultado = {
    "protocolo": "fase3_realizador", "criado_em": datetime.now().isoformat(), "commit": COMMIT,
    "base": str(BASE), "base_sha256": SHA_BASE, "pesos_sha256": sha256(DEST / "pesos.pt"),
    "hiperparametros": {"passos": PASSOS, "lote_sft": LOTE_SFT, "lote_replay": LOTE_REPLAY, "peso_replay": PESO_REPLAY,
                        "lr": LR, "aquecimento": AQUECIMENTO}, "melhor_passo_dev": melhor["passo"],
    "historico": historico, "baseline_blind": BASE_BLIND, "final_blind": FINAL,
    "perda_lm": {"base": PERDA_LM_BASE, "final": PERDA_LM_FINAL},
    "gates": GATES, "gate_pass": GATE_PASS,
    "porta_conversacional": "pendente", "ativo_no_chat": False, "promovido": False,
    "linhas_blind_final": FINAL_LINHAS, "linhas_blind_base": BASE_BLIND_LINHAS,
}
(DEST / "resultado_fase3.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=1), encoding="utf-8")
for p in (DEST / "pesos.pt", DEST / "tokenizer.json", DEST / "resultado_fase3.json"):
    if not p.is_file() or p.stat().st_size == 0:
        raise SystemExit(f"ERRO de persistência: {p}")
print(f"\nSALVO EM: {DEST}\ngate_pass={GATE_PASS} | porta conversacional pendente | ativo_no_chat=false | promovido=false", flush=True)
